## Functions

In [ ]:
%matplotlib inline
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items
import rpy2,os,re, shutil
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import seaborn as sns
import copy
sns.set(font="Arial")
sns.set_style("white")
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
from sklearn.decomposition import PCA as sklearnPCA
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pickle
#from statannot import add_stat_annotation
from matplotlib.gridspec import GridSpec


#rpy2
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri
rpy2.robjects.numpy2ri.activate()
from rpy2.robjects import pandas2ri
pandas2ri.activate()
import warnings
from rpy2.rinterface import RRuntimeWarning
warnings.filterwarnings("ignore", category=RRuntimeWarning)

from scipy.stats import zscore, spearmanr, ttest_ind, pearsonr,hypergeom, f_oneway
from scipy.interpolate import interp1d
from statsmodels.sandbox.stats.multicomp import multipletests
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import leidenalg
import igraph as ig
import gseapy as gp
from scipy.spatial import distance
from scipy.cluster.hierarchy import dendrogram, linkage, cut_tree
from sklearn.impute import KNNImputer
from scipy.io import loadmat
import itertools


class DESeq_Piano:
    def __init__(self,respath='.'):
        if respath=='.':
            print('Result Path is Not Defined, any result will be written in this directory')
            simpute_knn.respath=respath
        elif os.path.isdir(respath):
            print('Result Path Exists, please be careful of overwriting')
            self.respath=respath
        else:
            print('Result Path Does Not Exists, creating new directory')
            os.mkdir(respath)
            self.respath=respath
        self.__Init()
        
    def __Init(self):
        robjects.r('''
            library('DESeq2')
            library('piano')
            library('Biobase')
            library('snow')
            library('RColorBrewer')
            library('gplots')
            library('snowfall')
        ''')
        self.GSC_KEGG=None
        self.GSC_TF=None
        self.GSC_RM=None
        self.GSC_GO=None
        self.GSC_extraGSC=None
        self.KEGG=None
        self.TF=None
        self.RM=None
        self.GO=None
        self.extraGSC=None
        self.dds=None
        self.deseq_res=None
        self.conds=None
        self.count=None
        self.tpm=None

    def __check_dir(self,extrapath):
        if os.path.isdir('%s/%s/' % (self.respath,extrapath)):
            print('Path %s/%s/ exist, writing in the folder' % (self.respath,extrapath))
        else:
            print('Path %s/%s/ does not exist, making the folder to write the results' % (self.respath,extrapath))
            os.mkdir('%s/%s/' % (self.respath,extrapath))
            
    def DEseq(self,count_df=None,conds_series=None):
        if count_df is not None:
            self.count=count_df
        if conds_series is not None:
            self.conds=conds_series
        self.gene_names=count_df.index
        robjects.globalenv['conds']=np.array(self.conds)
        robjects.globalenv['subject']=np.array(self.conds.index)
        robjects.globalenv['count']=self.count.astype(int).values
        robjects.globalenv['gene_names']=np.array(self.gene_names)
        robjects.r('''
            conds=as.factor(conds)
            coldata <- data.frame(row.names=subject,conds)
            dds <- DESeqDataSetFromMatrix(countData=as.matrix(count),colData=coldata,design=~conds)
            dds <- DESeq(dds)
            print(resultsNames(dds))
            normalized_counts = counts(dds,normalized=TRUE)
            rownames(normalized_counts) = gene_names
            write.table(normalized_counts, file="../Data/normalized_counts.txt", sep="\t", quote=F, col.names=NA)
        ''')
        self.dds=robjects.globalenv['dds']
    
    def DEseq_Compare(self, cond1, cond2,save=True):
        if self.dds is None:
            raise ValueError('No DESeq data found, run the DEseq function')
        else:
            print('Comparing %s and %s' % (cond1,cond2))
            robjects.globalenv['dds']=self.dds
            robjects.globalenv['cond1']=cond1
            robjects.globalenv['cond2']=cond2
            robjects.globalenv['gene_names']=np.array(self.gene_names)
            robjects.r('''
                res=results(dds,contrast=c('conds',cond1,cond2))
                res=data.frame(res)
                res$rownames=gene_names
            ''')
            res = pd.DataFrame(robjects.globalenv['res']).set_index('rownames')
            self.deseq_df=res
            if save:
                self.__check_dir('deseq')
                res.to_csv('%s/deseq/deseq_%s_%s.txt' % (self.respath,cond1,cond2),sep='\t')
        
    def loadGSC(self,KEGG='',TF='',RM='',GO='',extraGSC=''):
        if KEGG != '':
            name='KEGG'
            self.__check_dir(name)
            self.GSC_KEGG=self.__loadGSC_execute(KEGG)
        if GO != '':
            name='GO'
            self.__check_dir(name)
            self.GSC_GO=self.__loadGSC_execute(GO)
        if RM != '':
            name='RM'
            self.__check_dir(name)
            self.GSC_RM=self.__loadGSC_execute(RM)
    
    def __loadGSC_execute(self,GSC):
        robjects.globalenv['GSC']=GSC 
        robjects.r('''
            y=loadGSC(GSC)
        ''')
        return robjects.globalenv['y']
    
    def __PIANO_execute(self,cond1,cond2,GSCtype='',save=True):
        robjects.globalenv['deseq_file']=pandas2ri.py2rpy_pandasdataframe(self.deseq_df)
        if GSCtype == 'KEGG':
            robjects.globalenv['y']=self.GSC_KEGG
            deseq_df=self.deseq_df
            deseq_df.index=[i.upper() for i in deseq_df.index]
            robjects.globalenv['deseq_file']=pandas2ri.py2rpy_pandasdataframe(deseq_df)
        elif GSCtype == 'GO':
            robjects.globalenv['y']=self.GSC_GO
            deseq_df=self.deseq_df
            deseq_df.index=[i.upper() for i in deseq_df.index]
            robjects.globalenv['deseq_file']=pandas2ri.py2rpy_pandasdataframe(deseq_df)
        elif GSCtype == 'RM':
            robjects.globalenv['y']=self.GSC_RM
            deseq_df=self.deseq_df
            deseq_df.index=[i.upper() for i in deseq_df.index]
            robjects.globalenv['deseq_file']=pandas2ri.py2rpy_pandasdataframe(deseq_df)
        robjects.r('''
            DESeq_file=deseq_file
            DESeq_file=DESeq_file[ ,c('log2FoldChange','pvalue')]
            logFC=as.matrix(DESeq_file[,1])
            pval=as.matrix(DESeq_file[,2])
            rownames(logFC)=(rownames(DESeq_file))
            rownames(pval)=(rownames(DESeq_file))
            logFC[is.na(logFC)] <- 0
            pval[is.na(pval)] <- 1
            gsaRes <- runGSA(pval,logFC,gsc=y, geneSetStat="reporter", signifMethod="nullDist", nPerm=1000,ncpus=8,gsSizeLim = c(5, Inf))
            res_piano=GSAsummaryTable(gsaRes)
            res_piano$rownames=rownames(res_piano)
        ''')
        res=pd.DataFrame(robjects.globalenv['res_piano']).set_index('rownames').set_index('Name').iloc[0:,0:]#.groupby('Name').sum()
        if save:
            res.to_csv('%s/%s/piano_%s_%s.txt' % (self.respath,GSCtype,cond1,cond2),sep='\t')
        return res

    def PIANO(self,cond1,cond2,save=True):
        if self.deseq_df is None:
            raise ValueError('No DEseq comparison found, run DEseq_Compare first or assign deseq result dataframe to deseq_res')
        else:
            if self.GSC_KEGG is not None:
                name='KEGG'
                self.KEGG = self.__PIANO_execute(cond1,cond2,GSCtype=name,save=save)
            if self.GSC_GO is not None:
                name='GO'
                self.GO = self.__PIANO_execute(cond1,cond2,GSCtype=name,save=save)         
            if self.GSC_RM is not None:
                name='RM'
                self.RM = self.__PIANO_execute(cond1,cond2,GSCtype=name,save=save)    
                    
    def __piano2xlsx(self,name,part='ALL'):
        if part != 'ALL':
            files=[i for i in sorted(os.listdir('%s/%s' % (self.respath,name))) if part in i]
        else:
            files=sorted(os.listdir('%s/%s' % (self.respath,name)))
        writer = pd.ExcelWriter('%s/%s_%s.xlsx' % (self.respath,name,part), engine='xlsxwriter')
        for i in files:
            if i.startswith('.'):
                continue
            deseq1=pd.read_csv('%s/%s/%s' % (self.respath,name,i),index_col='Name',sep='\t')[['Genes (tot)','Stat (dist.dir.up)','p (dist.dir.up)','p adj (dist.dir.up)']]
            deseq1.columns=['# of Genes', 'Stats','P-Value', 'P-Adj']
            deseq1=deseq1[deseq1['P-Value']<0.05]
            deseq1['Direction']='UP'
            deseq2=pd.read_csv('%s/%s/%s' % (self.respath,name,i),index_col='Name',sep='\t')[['Genes (tot)','Stat (dist.dir.up)','p (dist.dir.dn)','p adj (dist.dir.dn)']]
            deseq2.columns=['# of Genes', 'Stats','P-Value', 'P-Adj']
            deseq2=deseq2[deseq2['P-Value']<0.05]
            deseq2['Direction']='DOWN'
            deseq1=pd.concat([deseq1,deseq2])
            if len(i.replace('piano_','').replace('.txt',''))>31:
                print('sheet name too big, trimming to 30 character')
                n_temp=i.replace('piano_','').replace('.txt','')[0:31]
            else:
                n_temp=i.replace('piano_','').replace('.txt','')
            deseq1.sort_values('P-Value').to_excel(writer, sheet_name=n_temp)
        writer.close()
        
    def summarizePiano(self,names=['KEGG','GO','TF','RM','extraGSC'],part='ALL'):
        for name in names:
            if os.path.isdir(self.respath+name):
                self.__piano2xlsx(name,part=part)
            else:
                continue
    
    def summarizeDEseq(self,deseq_result='deseq',part='ALL'):
        if os.path.isdir(self.respath+deseq_result):
            if part != 'ALL':
                files=[i for i in sorted(os.listdir(self.respath+'/deseq/')) if part in i]
            else:
                files=sorted(os.listdir(self.respath+'/deseq/'))
            writer = pd.ExcelWriter(self.respath+'/DifferentialExpression_%s.xlsx' % part, engine='xlsxwriter')
            fin=0
            for i in files:
                if i.startswith('.'):
                    continue
                deseq=pd.read_csv(self.respath+'/deseq/%s' % i,index_col=0,sep='\t')
                deseq['abs']=deseq['log2FoldChange'].abs()
                deseq=deseq.sort_values('abs',ascending=False)[['log2FoldChange','pvalue','padj']]

                deseq['Direction']=['UP' if i > 0 else 'DOWN' for i in deseq['log2FoldChange']]
                deseq.columns=['L2FC','P-VALUE','P-ADJ','DIRECTION']
                deseq=deseq.sort_values('P-VALUE')
                if len(i.replace('deseq_','').replace('.txt',''))>31:
                    print('sheet name too big, trimming to 30 character')
                    n_temp=i.replace('deseq_','').replace('.txt','')[0:31]
                else:
                    n_temp=i.replace('deseq_','').replace('.txt','')
                deseq.to_excel(writer, sheet_name=n_temp)
            writer.close()
        else:
            raise ValueError('No DESEq result found')

    def save_object(self,filename='DEseq_PIANO.pkl'):
        with open('%s/%s' % (self.respath,filename), 'wb') as file:
            pickle.dump(self, file)
    
    def load_object(self,filename='DEseq_PIANO.pkl'):
        with open(filename, 'rb') as file:
            res=pickle.load(file)
        return res

def filter_tpm(tpm,conds,thr=1):
    #making sure that in at least 1 condition we have TPM > threshold (default 1)
    return tpm[(pd.concat([tpm.T,conds], axis = 1).groupby(conds.name).mean().T>thr).sum(1)>0]

def calc(tpm, clin = None,pval_thr=0.05,pos_only=False):
    print('Calculating Correlation..')
    if (clin) == None:
        tpm = tpm
    else:   
        tpm = pd.concat([tpm.T,clin], axis = 1).T
    temp=spearmanr(tpm.T)
    corr=pd.DataFrame(temp[0],columns=list(tpm.index),index=list(tpm.index))
    pval=pd.DataFrame(temp[1],columns=list(tpm.index),index=list(tpm.index))
    print('Filtering the matrix Correlation..')
    corr=corr.where(np.triu(np.ones(corr.shape)).astype(np.bool))
    pval=pval.where(np.triu(np.ones(pval.shape)).astype(np.bool))
    print('Making long table of Correlation..')
    corr2=corr.unstack().reset_index(name='weight')
    pval2=pval.unstack().reset_index(name='pval')
    res=corr2.merge(pval2,on=['level_0','level_1'])
    res=res[res['level_0'] != res['level_1']]
    res=res.dropna()
    print('Adjusting P-val')
    res['padj']=multipletests(res['pval'],method='fdr_bh')[1]
    res=res[res.padj < pval_thr].reset_index(drop=True)
    if pos_only:
        res=res[res['weight']>0]
    res=res[['level_0','level_1','weight']]
    print('Done!!')
    return res


class Network_Analysis:
    def __init__(self,filename, network_type,respath,KEGG='',GO='',topx=0.05,cluster_size=30):
        self.network_ori=pd.read_csv(filename,sep='\t')
        self.res_path=respath
        if os.path.isdir(self.res_path):
            pass
        else:
            os.mkdir(self.res_path)
        self.cluster_size=cluster_size
        self.topx=topx
        if network_type == 'transcriptomics':
            self.__net_analysis_transcriptomics()
        else:
            self.__net_analysis_otheromics()
        
    def __net_analysis_transcriptomics(self):
        print('Loading The Network...')
        temp=self.network_ori[self.network_ori['weight']>0]
        temp=temp[temp['weight']>temp['weight'].quantile(1-self.topx)]
        g= ig.Graph.TupleList(zip(temp['level_0'],temp['level_1']))
        print('Cluster Analysis...')
        optimiser = leidenalg.Optimiser()
        clust_calc = leidenalg.ModularityVertexPartition(g)
        diff =  optimiser.optimise_partition(clust_calc, n_iterations=-1)
        clust=pd.Series(clust_calc.membership,index=g.vs['name'])
        thr=self.cluster_size
        temp_c=clust.value_counts()[clust.value_counts()>thr].index.tolist()
        self.degree = pd.Series(g.degree(), index = g.vs['name'], name = 'degree')
        self.clustering=clust[clust.isin(temp_c)]
        self.modularity=clust_calc.modularity
        self.network=g
        temp=self.network_ori.merge(pd.DataFrame(self.clustering).reset_index(),how='left',left_on='level_0',right_on='index').merge(pd.DataFrame(self.clustering).reset_index(),how='left',left_on='level_1',right_on='index').dropna()
        temp2=temp[['0_x','0_y']]
        temp2['count']=1
        temp2=temp2.groupby(['0_x','0_y']).count().reset_index().pivot_table(index='0_x',columns='0_y',values='count').fillna(0)
        temp2=pd.DataFrame([(i,j,(temp2.loc[i,j]+temp2.loc[j,i]),(temp2.loc[i,j]+temp2.loc[j,i])/(self.clustering.value_counts()[i]+self.clustering.value_counts()[j])) for i in temp2.index for j in temp2.columns if (i < j)],columns=['Source','Target','Weight1','Weight2'])
        temp2.astype(int).to_csv('%s/edge_pos_cluster.txt' % self.res_path,sep='\t')
        pd.DataFrame(self.clustering.value_counts(),columns=['Size']).to_csv('%s/node_pos_cluster.txt' % self.res_path,sep='\t')
        pd.concat([self.clustering.rename('cluster'),  self.degree.reindex(self.clustering.index)], axis = 1).to_csv('%s/clust_pos_cluster.txt' % self.res_path,sep='\t')

        
    def __net_analysis_otheromics(self):
        print('Loading The Network...')
        temp=self.network_ori
        temp=temp[(temp['weight']>temp['weight'].quantile(1-self.topx)) | (temp['weight']<temp['weight'].quantile(self.topx))]
        g= ig.Graph.TupleList(zip(temp['level_0'],temp['level_1'],temp['weight']),weights=True)
        print('Cluster Analysis...')
        G_pos = g.subgraph_edges(g.es.select(weight_gt = 0), delete_vertices=False)
        G_neg = g.subgraph_edges(g.es.select(weight_lt = 0), delete_vertices=False)
        G_neg.es['weight'] = [-w for w in G_neg.es['weight']]
        part_pos = leidenalg.ModularityVertexPartition(G_pos, weights='weight')
        part_neg = leidenalg.ModularityVertexPartition(G_neg, weights='weight');
        optimiser = leidenalg.Optimiser()
        diff = optimiser.optimise_partition_multiplex([part_pos, part_neg],layer_weights=[1,-1], n_iterations=-1)
        clust = pd.DataFrame(pd.Series(part_pos.membership+part_neg.membership,index=G_pos.vs['name']+G_neg.vs['name'])).reset_index().drop_duplicates().set_index('index')[0]
        thr=self.cluster_size
        temp_c=clust.value_counts()[clust.value_counts()>thr].index.tolist()
        self.degree_combi = pd.Series(g.degree(), index = g.vs['name'], name = 'degree')
        self.clustering_combi=clust[clust.isin(temp_c)]
        self.modularity_combi=diff
        self.network_combi=g
        
        temp=self.network_ori.merge(pd.DataFrame(self.clustering_combi).reset_index(),how='left',left_on='level_0',right_on='index').merge(pd.DataFrame(self.clustering_combi).reset_index(),how='left',left_on='level_1',right_on='index').dropna()
        temp2=temp[['0_x','0_y']]
        temp2['count'] = 1
        temp2=temp2.groupby(['0_x','0_y']).count().reset_index().pivot_table(index='0_x',columns='0_y',values='count').fillna(0)
        temp2=pd.DataFrame([(i,j,(temp2.loc[i,j]+temp2.loc[j,i]),(temp2.loc[i,j]+temp2.loc[j,i])/(self.clustering_combi.value_counts()[i]+self.clustering_combi.value_counts()[j])) for i in temp2.index for j in temp2.columns if (i < j)],columns=['Source','Target','Weight1','Weight2'])
        temp2.astype(int).to_csv('%s/edge_combi_cluster.txt' % self.res_path,sep='\t')
        pd.DataFrame(self.clustering_combi.rename('Size').value_counts(),columns=['Size']).to_csv('%s/node_combi_cluster.txt' % self.res_path,sep='\t')
        pd.concat([self.clustering_combi.rename('cluster'), self.degree_combi.reindex(self.clustering_combi.index)], axis = 1).to_csv('%s/clust_combi_cluster.txt' % self.res_path,sep='\t')

    
    def save_network(self):
        pickle_out = open('%s/network_object.pkl' % self.res_path,"wb")
        pickle.dump(self, pickle_out)
        pickle_out.close()
        
def impute_knn(data):
    imputer = KNNImputer(n_neighbors=2)
    After_imputation = imputer.fit_transform(data)
    return pd.DataFrame(After_imputation, index = data.index, columns = data.columns)

## Data Loading

In [ ]:
supp_data = pd.ExcelFile('Data/Data S1.xlsx')

metadata = supp_data.parse('Condition', index_col = 0)

## Clinical Data Statistics

(output in Results_Paper/ClinicalMeasurementStatistics.txt)

In [ ]:
data = supp_data.parse('Clinical', index_col = 0)
data = data.merge(metadata[['Code', 'Condition']].drop_duplicates(), left_index = True, right_on = 'Code')[list(data.columns) + ['Condition']]


In [ ]:
table1 = pd.DataFrame()
for i in data.columns[1:-1]:
    temp = data[[i, 'Condition']].dropna()
    mean = temp.groupby('Condition')[i].mean()
    sem = temp.groupby('Condition')[i].sem()
    fin = (mean.apply(lambda x: "%.2f" % x) + u" \u00B1 " +sem.apply(lambda x: "%.2f" % x))[['Ctrl', 'Ischemia', 'Both']]
    
    if f_oneway(temp[temp['Condition'] == 'Ctrl'][i].tolist(), 
                temp[temp['Condition'] == 'Ischemia'][i].tolist(),
                temp[temp['Condition'] == 'Both'][i].tolist())[1] < 0.05:
        tukey = pairwise_tukeyhsd(endog=temp[i],groups=temp['Condition']).pvalues
        if tukey[2] <= 0.0005:
            sig = '***'
        elif tukey[2] <= 0.005:
            sig = '**'
        elif tukey[2] <= 0.05:
            sig = '*'
        else:
            sig = 'ns'
        fin['P-Adj\n(Ischemia vs Ctrl)'] = sig
        
        
        if tukey[0] <= 0.0005:
            sig = '***'
        elif tukey[0] <= 0.005:
            sig = '**'
        elif tukey[0] <= 0.05:
            sig = '*'
        else:
            sig = 'ns'
        fin['P-Adj\n(Both vs Ctrl)'] = sig
    
        if tukey[1] <= 0.0005:
            sig = '***'
        elif tukey[1] <= 0.005:
            sig = '**'
        elif tukey[1] <= 0.05:
            sig = '*'
        else:
            sig = 'ns'
        fin['P-Adj\n(Both vs Ischemia)'] = sig

    table1 = pd.concat([table1, fin], axis = 1).fillna('ns')
table1.T.to_csv('Results_Paper/ClinicalMeasurementStatistics.txt', sep = "\t")

## Lifestyle

In [ ]:
data = supp_data.parse('Lifestyle', index_col = 0)
data = data.merge(metadata[['Code', 'Condition']].drop_duplicates(), left_index = True, right_on = 'Code')[list(data.columns) + ['Condition']]
data['count'] = 1

In [ ]:
for i in ['Smoking', 'Alcohol', 'Snus']:
    total = data.groupby(['Condition', i])['count'].sum()
    percentage = data.groupby(['Condition', i])['count'].sum()/data.groupby(['Condition'])['count'].sum()
    
    print((total.apply(lambda x: "%d" % x) + " (" +percentage.apply(lambda x: "%.2f%%" % x) + ')')[['Ctrl', 'Ischemia', 'Both']])
    print("-----------")

## CVD Comorbidities 

(output in Results_Paper/CVD_Comorbidities.txt)

In [ ]:
data = supp_data.parse('Other', index_col = 0).replace('Yes', 1).replace('No', np.nan)
data = data.T[data.sum() > 0].T
data = data.merge(metadata[['Code', 'Condition']].drop_duplicates(), left_index = True, right_on = 'Code')[list(data.columns) + ['Condition']]
data['count'] = 1

In [ ]:
total = data.groupby('Condition').sum().reindex(['Ctrl', 'Ischemia', 'Both'])
percentage = data.groupby(['Condition']).sum()/data.fillna(1).groupby(['Condition']).sum()

(total.apply(lambda x: ["%d" % i for i in x]) + " (" + percentage.apply(lambda x: ["%.2f%%" % i for i in x]) + ')').replace('0 (0.00%)', '0').T[['Ctrl', 'Ischemia', 'Both']].to_csv('Results_Paper/CVD_Comorbidities.txt', sep = '\t')

In [ ]:
total

## Medication

Results_Paper/Medication_Summary.txt

In [ ]:
data = supp_data.parse('Medication', index_col = 0).replace('Yes', 1).replace('No', np.nan).iloc[0:,0:-1]
data = data.T[data.sum() > 0].T
data = data.merge(metadata[['Code', 'Condition']].drop_duplicates(), left_index = True, right_on = 'Code')[list(data.columns) + ['Condition']]


In [ ]:
total = data.groupby('Condition').sum().reindex(['Ctrl', 'Ischemia', 'Both'])
percentage = data.groupby(['Condition']).sum()/data.fillna(1).groupby(['Condition']).sum()

(total.apply(lambda x: ["%d" % i for i in x]) + " (" + percentage.apply(lambda x: ["%.2f%%" % i for i in x]) + ')').replace('0 (0.00%)', '0').T[['Ctrl', 'Ischemia', 'Both']].to_csv('Results_Paper/Medication_Summary.txt', sep = '\t')

## Differential Expression Analysis

DESeq2 output: Results_Paper/deseq

KEGG Enrichment output: Results_Paper/KEGG

GO Enrichment output: Results_Paper/GO

Summary of each analysis can be found in the same folders

In [ ]:
metadata_x = metadata.copy()
metadata_x['Condition'] = metadata['Condition'] + '_' + metadata['Tissue']
count = supp_data.parse('Count', index_col = 0)[metadata_x.index]



res_path='Results_Paper/'
k=DESeq_Piano(res_path)
k.DEseq(count,metadata_x['Condition'])
k.loadGSC(KEGG='Library/KEGG_2021_Human.gmt', GO = 'Library/GO_Biological_Process_2023.gmt')

In [ ]:
cond2='Ctrl_EF'
cond1='Both_EF'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

cond2='Ctrl_EF'
cond1='Ischemia_EF'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

cond2='Ctrl_LV'
cond1='Both_LV'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

cond2='Ctrl_LV'
cond1='Ischemia_LV'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

cond2='Ischemia_EF'
cond1='Both_EF'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

cond2='Ischemia_LV'
cond1='Both_LV'
k.DEseq_Compare(cond1,cond2,save=True)
k.PIANO(cond1,cond2,save=True)

k.summarizeDEseq()
k.summarizePiano()

## Figure 1A

In [ ]:
deseq = pd.ExcelFile('Results_Paper/DifferentialExpression_ALL.xlsx')

In [ ]:
res = pd.DataFrame()
lst = ['Ischemia_LV_Ctrl_LV', 'Both_LV_Ctrl_LV', 'Both_LV_Ischemia_LV']
for i in lst:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['DIRECTION'].rename(i.replace('_LV', ''))
    res = pd.concat([res, temp], axis = 1)



In [ ]:
temp = res.unstack().reset_index().dropna().groupby(['level_0',0]).count().reset_index()

In [ ]:
plt.figure(figsize = (4.5,4))
g = sns.barplot(data = temp, x = 'level_0', y = 'level_1', hue = 0, order = ['Ischemia_Ctrl', 'Both_Ctrl', 'Both_Ischemia'], hue_order = ['UP', 'DOWN'], 
               palette = ['#a50000','#0000a5'])
# g.set_yticklabels([int(i) for i in g.get_yticks()], fontsize = 15)
g.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.get_xticklabels()],rotation=0, ha = 'center', fontsize = 15, fontweight = 'bold')
g.set_xlabel('')
g.set_ylabel('# of DEGs', fontsize= 20, fontweight = 'bold')
g.get_legend().set_title('Direction')
plt.setp(g.get_legend().get_title(), fontsize='15', fontweight = 'bold')
plt.setp(g.get_legend().get_texts(), fontsize='15')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15),
          fancybox=True, ncol=5)
plt.tight_layout()

plt.savefig('Results_Paper/FiguresPaper/1A.pdf')

## Figure 1B-C

In [ ]:
temp = open('Library/ko00001.keg', 'r').readlines()
x = []
for i in temp:
    if i.startswith('A'):
        A = ' '.join(i.split(' ')[1:]).rstrip('\n')
    if i.startswith('B  '):
        B = ' '.join(i.replace('B  ', '').split(' ')[1:]).rstrip('\n')
    if i.startswith('C    '):
        C = ' '.join(i.replace('C    ', '').split(' ')[1:]).split(' [PATH')[0].rstrip('\n')
        x.append([A, B, C])
x = pd.DataFrame(x, columns = ['A', 'B', 'C'])

exclude_KEGG = ['Hepatitis B', 'Human T-cell leukemia virus 1 infection', 'Regulation of actin cytoskeleton', 'Measles',
           'MicroRNAs in cancer', 'Pathways in cancer', 'Influenza A', 'Oocyte meiosis', 'Progesterone-mediated oocyte maturation',
           'Ribosome biogenesis in eukaryotes', 'Amoebiasis', 'Small cell lung cancer', 'Epstein-Barr virus infection', 'Human papillomavirus infection',
           'Pertussis', 'Staphylococcus aureus infection', 'Proteoglycans in cancer', 'Leishmaniasis', 'Legionellosis', 'Viral carcinogenesis', 
           'Chagas disease (American trypanosomiasis)', 'Toxoplasmosis', 'Malaria', 'Metabolism of xenobiotics by cytochrome P450', 'Chronic myeloid leukemia',
           'Prion diseases', 'Inflammatory bowel disease (IBD)', 'Kaposi sarcoma-associated herpesvirus infection', 'Parkinson disease', 'Hepatitis C', 'Systemic lupus erythematosus',
           'Tuberculosis', 'Hepatitis B', 'Measles', 'Hematopoietic cell lineage', 
           'Epstein-Barr virus infection', 'Drug metabolism', 'Salivary secretion', 'Glioma', 'Human cytomegalovirus infection', 'Melanoma', 'Chemical carcinogenesis',
            'African trypanosomiasis', 'Collecting duct acid secretion', 'African trypanosomiasis', 
            'Rheumatoid arthritis', 'Metabolic pathways', 'Spinocerebellar ataxia', 'Drug metabolism - other enzymes', 'Asthma',
                'Viral protein interaction with cytokine and cytokine receptor','Type II diabetes mellitus', 'Acute myeloid leukemia', 'Caffeine metabolism', 'Central carbon metabolism in cancer',
                'Bladder cancer','Hepatocellular carcinoma', 'Axon guidance', 'Fanconi anemia pathway', 'Inflammatory bowel disease', 'Neuroactive ligand-receptor interaction'
          ]

In [ ]:
temp = pd.ExcelFile('Results_Paper/KEGG_ALL.xlsx')
lst = ['Ischemia_LV_Ctrl_LV', 'Both_LV_Ctrl_LV', 'Both_LV_Ischemia_LV']
res = []
for i in lst:
    ha = temp.parse(i, index_col = 0)
    ha = ha[ha['P-Value'] < 0.01]
    res.append(ha['Stats'].rename(i.replace('_LV', '')))
res = pd.concat(res, axis = 1)

# Excluding the unrelated diseases and pathways
res = res[~(res.index.str.contains('isease') | res.index.str.contains('nfection') | res.index.str.contains('ancer')| res.index.str.contains('epatitis')
           | res.index.str.contains('Malaria')  | res.index.str.contains('Gastric')  | res.index.str.contains('Influenza')
            | res.index.str.contains('Leishmaniasis') | res.index.str.contains('Viral') | res.index.str.contains('clerosi')| res.index.str.contains('Ribo')
            | res.index.isin(exclude_KEGG) | res.index.str.contains('higel') | res.index.str.contains('mmuno') | res.index.str.contains('leuke') | res.index.str.contains('Melanogenesis')
           )]#.merge(x, left_index = True, right_on = 'C').sort_values(['A', 'B']).set_index('C')
res['B'] = np.nan
res.loc[res['Ischemia_Ctrl'].notna() & res['Both_Ctrl'].notna(), 'B'] = 'Shared'
res.loc[res['Ischemia_Ctrl'].notna() & res['Both_Ctrl'].isna(), 'B'] = 'Ischemia'
res.loc[res['Ischemia_Ctrl'].isna() & res['Both_Ctrl'].notna(), 'B'] = 'Both'

In [ ]:
temp

In [ ]:
comp = 'Ischemia_Ctrl'
plt.figure(figsize = (3, 2.5))
temp = res[res['B'] == 'Ischemia'].reset_index().sort_values(comp)
temp.loc[temp[comp] < 0,'B'] = '#0000a5'
temp.loc[temp[comp] > 0,'B'] = '#a50000'
ax = sns.barplot(data = temp, x = comp, y = 'Name')
ylim = ax.get_ylim()
ax.vlines([0], ymin = ylim[1], ymax = ylim[0], color = 'k')
ax.set_ylim(ylim[1], ylim[0])
ax.set_ylabel('')
ax.set_xlabel('Gene-Level Statistics\n(Ischemia vs Ctrl)', fontweight = 'bold')
# ax.spines['left'].set_position('zero')
ax.set_title(comp, fontsize = 10, fontweight = 'bold')

sns.despine(ax = ax, left = True)
plt.savefig('Results_Paper/FiguresPaper/1B.pdf')

In [ ]:
res[['Ischemia_Ctrl', 'Both_Ctrl']].min().min()

In [ ]:
cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#0000a5",'#0000d8',"#FFFAF0",'#d80000',"#a50000"])

temp = res[res['Both_Ischemia'].notna() | (res['Both_Ctrl'].notna()) | (res['Both_Ctrl'].notna() & res['Ischemia_Ctrl'].notna())]
g = sns.clustermap(temp.fillna(0)[['Ischemia_Ctrl', 'Both_Ctrl', 'Both_Ischemia']], yticklabels = 1, figsize = (3,10),
               center = 0, cmap = cmap, row_cluster=True, col_cluster=False,
                vmin = res[['Ischemia_Ctrl', 'Both_Ctrl']].min().min(), vmax = res[['Ischemia_Ctrl', 'Both_Ctrl']].max().max(),
               cbar_kws={"orientation": "vertical", 'label': 'Gene-Level Statistics'},
              )
# g.ax_heatmap.hlines(range(1, len(temp.index)), xmin = g.ax_heatmap.get_xlim()[1], xmax = g.ax_heatmap.get_xlim()[0], color = 'k', linewidths = 0.2)
g.ax_heatmap.vlines([len(temp.columns)-1], ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'white')
g.ax_heatmap.vlines(range(1, len(temp.columns)-1), ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'k')
# g.ax_heatmap.set_title(i, fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.ax_heatmap.get_xticklabels()],rotation=0, ha = 'center', fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize = 10,rotation=0)
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/1C.pdf')

## Figure 2A

In [ ]:
deseq = pd.ExcelFile('Results_Paper/DifferentialExpression_ALL.xlsx')

In [ ]:
res = pd.DataFrame()
lst = ['Ischemia_EF_Ctrl_EF', 'Both_EF_Ctrl_EF', 'Both_EF_Ischemia_EF']
for i in lst:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['DIRECTION'].rename(i.replace('_EF', ''))
    res = pd.concat([res, temp], axis = 1)



In [ ]:
temp = res.unstack().reset_index().dropna().groupby(['level_0',0]).count().reset_index()

In [ ]:
plt.figure(figsize = (3,4))
g = sns.barplot(data = temp, x = 'level_0', y = 'level_1', hue = 0, order = res.columns, hue_order = ['UP', 'DOWN'], 
               palette = ['#a50000','#0000a5'])
g.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.get_xticklabels()],rotation=0, ha = 'center', fontsize = 15, fontweight = 'bold')
g.set_xlabel('')
g.set_ylabel('# of DEGs', fontsize= 20, fontweight = 'bold')
g.get_legend().set_title('Direction')
plt.setp(g.get_legend().get_title(), fontsize='15', fontweight = 'bold')
plt.setp(g.get_legend().get_texts(), fontsize='15')
plt.legend(loc='upper center', bbox_to_anchor=(0.5, 1.15),
          fancybox=True, ncol=5)
# g.set_yticklabels([int(i) for i in g.get_yticks()], fontsize = 15)

plt.tight_layout()

plt.savefig('Results_Paper/FiguresPaper/2A.pdf')

## Figure 2B-F

In [ ]:
temp = open('Library/ko00001.keg', 'r').readlines()
x = []
for i in temp:
    if i.startswith('A'):
        A = ' '.join(i.split(' ')[1:]).rstrip('\n')
    if i.startswith('B  '):
        B = ' '.join(i.replace('B  ', '').split(' ')[1:]).rstrip('\n')
    if i.startswith('C    '):
        C = ' '.join(i.replace('C    ', '').split(' ')[1:]).split(' [PATH')[0].rstrip('\n')
        x.append([A, B, C])
x = pd.DataFrame(x, columns = ['A', 'B', 'C'])

exclude_KEGG = ['Hepatitis B', 'Human T-cell leukemia virus 1 infection', 'Regulation of actin cytoskeleton', 'Measles',
           'MicroRNAs in cancer', 'Pathways in cancer', 'Influenza A', 'Oocyte meiosis', 'Progesterone-mediated oocyte maturation',
           'Ribosome biogenesis in eukaryotes', 'Amoebiasis', 'Small cell lung cancer', 'Epstein-Barr virus infection', 'Human papillomavirus infection',
           'Pertussis', 'Staphylococcus aureus infection', 'Proteoglycans in cancer', 'Leishmaniasis', 'Legionellosis', 'Viral carcinogenesis', 
           'Chagas disease (American trypanosomiasis)', 'Toxoplasmosis', 'Malaria', 'Metabolism of xenobiotics by cytochrome P450', 'Chronic myeloid leukemia',
           'Prion diseases', 'Inflammatory bowel disease (IBD)', 'Kaposi sarcoma-associated herpesvirus infection', 'Parkinson disease', 'Hepatitis C', 'Systemic lupus erythematosus',
           'Tuberculosis', 'Hepatitis B', 'Measles', 'Hematopoietic cell lineage', 
           'Epstein-Barr virus infection', 'Drug metabolism', 'Salivary secretion', 'Glioma', 'Human cytomegalovirus infection', 'Melanoma', 'Chemical carcinogenesis',
            'African trypanosomiasis', 'Collecting duct acid secretion', 'African trypanosomiasis', 
            'Rheumatoid arthritis', 'Metabolic pathways', 'Spinocerebellar ataxia', 'Drug metabolism - other enzymes', 'Asthma',
                'Viral protein interaction with cytokine and cytokine receptor','Type II diabetes mellitus', 'Acute myeloid leukemia', 'Caffeine metabolism', 'Central carbon metabolism in cancer',
                'Bladder cancer','Hepatocellular carcinoma', 'Axon guidance', 'Fanconi anemia pathway', 'Inflammatory bowel disease'
          ]

In [ ]:
temp = pd.ExcelFile('Results_Paper/KEGG_ALL.xlsx')
lst = ['Ischemia_EF_Ctrl_EF', 'Both_EF_Ctrl_EF', 'Both_EF_Ischemia_EF']
res = []
for i in lst:
    ha = temp.parse(i, index_col = 0)
    ha = ha[ha['P-Adj'] < 0.005]
    res.append(ha['Stats'].rename(i.replace('_EF', '')))
res = pd.concat(res, axis = 1)

# Excluding the unrelated diseases and pathways
res = res[~(res.index.str.contains('isease') | res.index.str.contains('nfection') | res.index.str.contains('ancer')| res.index.str.contains('epatitis')
           | res.index.str.contains('Malaria')  | res.index.str.contains('Gastric')  | res.index.str.contains('Influenza')
            | res.index.str.contains('Leishmaniasis') | res.index.str.contains('Viral') | res.index.str.contains('clerosi')| res.index.str.contains('Ribo')
            | res.index.isin(exclude_KEGG) | res.index.str.contains('higel') | res.index.str.contains('mmuno') | res.index.str.contains('leuke') | res.index.str.contains('Intest')
             | res.index.str.contains('Olfact')
           )]#.merge(x, left_index = True, right_on = 'C').sort_values(['A', 'B']).set_index('C')
res['B'] = np.nan
res.loc[res['Ischemia_Ctrl'].notna() & res['Both_Ctrl'].notna(), 'B'] = 'Shared'
res.loc[res['Ischemia_Ctrl'].notna() & res['Both_Ctrl'].isna(), 'B'] = 'Ischemia'
res.loc[res['Ischemia_Ctrl'].isna() & res['Both_Ctrl'].notna(), 'B'] = 'Both'

In [ ]:
cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#0000a5",'#0000d8',"#FFFAF0",'#d80000',"#a50000"])

temp = res[res['Both_Ischemia'].notna() | (res['Both_Ctrl'].notna()) | (res['Both_Ctrl'].notna() & res['Ischemia_Ctrl'].notna())]
g = sns.clustermap(temp.fillna(0)[['Ischemia_Ctrl', 'Both_Ctrl', 'Both_Ischemia']], yticklabels = 1, figsize = (3,15),
               center = 0, cmap = cmap, row_cluster=True, col_cluster=False,
                vmin = res[['Ischemia_Ctrl', 'Both_Ctrl']].min().min(), vmax = res[['Ischemia_Ctrl', 'Both_Ctrl']].max().max(),
               cbar_kws={"orientation": "vertical", 'label': 'Gene-Level Statistics'},
              )
# g.ax_heatmap.hlines(range(1, len(temp.index)), xmin = g.ax_heatmap.get_xlim()[1], xmax = g.ax_heatmap.get_xlim()[0], color = 'k', linewidths = 0.2)
g.ax_heatmap.vlines([len(temp.columns)-1], ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'white')
g.ax_heatmap.vlines(range(1, len(temp.columns)-1), ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'k')
# g.ax_heatmap.set_title(i, fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.ax_heatmap.get_xticklabels()],rotation=0, ha = 'center', fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize = 10,rotation=0)
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/2B.pdf')

## Figure S2

In [ ]:
temp = pd.ExcelFile('Results_Paper/GO_ALL.xlsx')
res = []
lst = ['Ischemia_LV_Ctrl_LV', 'Both_LV_Ctrl_LV']

for i in lst:
    ha = temp.parse(i, index_col = 0)
    ha = ha[ha['P-Value'] < 0.01]
    res.append(ha['Stats'].rename(i).replace('_LV', ''))
res = pd.concat(res, axis = 1)

# Excluding the unrelated diseases and pathways
res = res[~(res.index.str.contains('isease') | res.index.str.contains('nfection') | res.index.str.contains('ancer')| res.index.str.contains('epatitis')
           | res.index.str.contains('Malaria')  | res.index.str.contains('Gastric')  | res.index.str.contains('Influenza')
            | res.index.str.contains('Leishmaniasis') | res.index.str.contains('Viral') | res.index.str.contains('clerosi')| res.index.str.contains('Ribo')
            | res.index.str.contains('higel') | res.index.str.contains('mmuno') | res.index.str.contains('leuke')
           )].dropna(how = 'all')

res = res[res.columns[(~res.columns.str.contains('Diabete'))]]

In [ ]:
cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#0000a5",'#0000d8',"#FFFAF0",'#d80000',"#a50000"])

temp = res[(~res.index.str.contains('positive reg')) & (~res.index.str.contains('negative reg'))].dropna()

g = sns.clustermap(temp.fillna(0),#[['Ischemia_Ctrl', 'Both_Ctrl']], 
                   yticklabels = 1, figsize = (2,12),
               center = 0, cmap = cmap, row_cluster=True, col_cluster=False,
                # vmin = res.min()[['Ischemia_Ctrl', 'Both_Ctrl']].min(), vmax = res.max()[['Ischemia_Ctrl', 'Both_Ctrl']].max(),
               cbar_kws={"orientation": "vertical", 'label': 'Gene-Level Statistics'},
              )
g.ax_heatmap.hlines(range(1, len(temp.index)), xmin = g.ax_heatmap.get_xlim()[1], xmax = g.ax_heatmap.get_xlim()[0], color = 'k', linewidths = 0.2)
g.ax_heatmap.vlines([len(temp.columns)-1], ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'white')
g.ax_heatmap.vlines(range(1, len(temp.columns)-1), ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'k')
# g.ax_heatmap.set_title(i, fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.ax_heatmap.get_xticklabels()],rotation=0, ha = 'center', fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize = 10,rotation=0)
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/S2.pdf')

## Figure S3

In [ ]:
temp = pd.ExcelFile('Results_Paper/GO_ALL.xlsx')
res = []
lst = ['Ischemia_EF_Ctrl_EF', 'Both_EF_Ctrl_EF']

for i in lst:
    ha = temp.parse(i, index_col = 0)
    ha = ha[ha['P-Value'] < 0.01]
    res.append(ha['Stats'].rename(i).replace('_EF', ''))
res = pd.concat(res, axis = 1)

# res = res[res.index.isin(classification['name'])]


# Excluding the unrelated diseases and pathways
res = res[~(res.index.str.contains('isease') | res.index.str.contains('nfection') | res.index.str.contains('ancer')| res.index.str.contains('epatitis')
           | res.index.str.contains('Malaria')  | res.index.str.contains('Gastric')  | res.index.str.contains('Influenza')
            | res.index.str.contains('Leishmaniasis') | res.index.str.contains('Viral') | res.index.str.contains('clerosi')| res.index.str.contains('Ribo')
            | res.index.str.contains('higel') | res.index.str.contains('mmuno') | res.index.str.contains('leuke')
           )].dropna(how = 'all')

res = res[res.columns[(~res.columns.str.contains('Diabete'))]]

In [ ]:
cmap=matplotlib.colors.LinearSegmentedColormap.from_list("", ["#0000a5",'#0000d8',"#FFFAF0",'#d80000',"#a50000"])

temp = res[(~res.index.str.contains('positive reg')) & (~res.index.str.contains('negative reg'))]
temp = temp[temp.columns[temp.columns.str.contains('EF')]].dropna()
g = sns.clustermap(temp.fillna(0),#[['Ischemia_Ctrl', 'Both_Ctrl']], 
                   yticklabels = 1, figsize = (2,21),
               center = 0, cmap = cmap, row_cluster=True, col_cluster=False,
                # vmin = res.min()[['Ischemia_Ctrl', 'Both_Ctrl']].min(), vmax = res.max()[['Ischemia_Ctrl', 'Both_Ctrl']].max(),
               cbar_kws={"orientation": "vertical", 'label': 'Gene-Level Statistics'},
              )
g.ax_heatmap.hlines(range(1, len(temp.index)), xmin = g.ax_heatmap.get_xlim()[1], xmax = g.ax_heatmap.get_xlim()[0], color = 'k', linewidths = 0.2)
g.ax_heatmap.vlines([len(temp.columns)-1], ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'white')
g.ax_heatmap.vlines(range(1, len(temp.columns)-1), ymin = g.ax_heatmap.get_ylim()[1], ymax = g.ax_heatmap.get_ylim()[0], color = 'k')
# g.ax_heatmap.set_title(i, fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_xticklabels([i.get_text().replace('_', '\nvs\n') for i in g.ax_heatmap.get_xticklabels()],rotation=0, ha = 'center', fontsize = 10, fontweight = 'bold')
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize = 10,rotation=0)
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/S3.pdf')

## CO-Expression Network Generation (Figure 3A, B)

Network was visualized using cytoscape, with the cluster clustering coefficient as the node size and relative expected edges (weight2) as the edge weight.

In [ ]:
tpm_all = supp_data.parse('TPM', index_col = 0)

In [ ]:
tissue = 'LV'
temp_metadata = (metadata[(metadata['Tissue'] == tissue)])
temp_tpm = tpm_all.loc[(~tpm_all.index.duplicated()), temp_metadata.index]

# sel = filter_tpm(temp_tpm, temp_metadata['conds'], thr = 1)

temp = pd.concat([temp_tpm.T, temp_metadata['Condition']], axis = 1).groupby('Condition').mean()
temp = temp.T[temp.std() > temp.std().quantile(.25)].index.tolist()
tpm_filt = temp_tpm.reindex(temp)

# coexp = calc(tpm_filt, pval_thr = 0.05)
# coexp.to_csv('Results_Paper/GCN_%s.txt' % tissue, sep = '\t', index = False)

topx = 0.25
k=Network_Analysis(filename='Results_Paper/GCN_%s.txt' % tissue, network_type='transcriptomics',respath='Results_Paper/GCN_%s/' % tissue,
                   topx=topx, cluster_size = 30)
k.save_network()

In [ ]:
tissue = 'LV'
k = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))
color_mapping = dict(zip(sorted(k.clustering.unique()),sns.color_palette("tab10").as_hex()[::-1]))
x = []
for cl in sorted(k.clustering.unique()):
    k_temp = k.network.subgraph(k.clustering[k.clustering == cl].index)
    print(cl)
    print(k_temp.transitivity_avglocal_undirected())
    x.append(['%s-%d'% (tissue,cl), k_temp.transitivity_avglocal_undirected(), color_mapping[cl]])
x = pd.DataFrame(x)

plt.figure(figsize = (3,5))
ax = sns.barplot(data = x, x = 0, y = 1, palette = x[2])
ax.set_ylabel('Average Clustering Coefficients', fontsize = 15, fontweight = 'bold')
ax.set_xlabel('', fontsize = 15, fontweight = 'bold')
ax.set_xticklabels(ax.get_xticklabels(), fontsize = 15, fontweight = 'bold', rotation = 90, ha = 'center', va = 'top')
sns.despine(ax = ax)

x.to_csv('Results_Paper/GCN_%s/ClusteringCoefficient.txt' % tissue, sep = '\t')

plt.savefig('Results_Paper/GCN_%s/ClusteringCoefficient.pdf' % tissue, transparent = True)

In [ ]:
tissue = 'EF'
temp_metadata = (metadata[(metadata['Tissue'] == tissue)])
temp_tpm = tpm_all.loc[(~tpm_all.index.duplicated()), temp_metadata.index]

# sel = filter_tpm(temp_tpm, temp_metadata['conds'], thr = 1)

temp = pd.concat([temp_tpm.T, temp_metadata['Condition']], axis = 1).groupby('Condition').mean()
temp = temp.T[temp.std() > temp.std().quantile(.25)].index.tolist()
tpm_filt = temp_tpm.reindex(temp)

# coexp = calc(tpm_filt, pval_thr = 0.05)
# coexp.to_csv('Results_Paper/GCN_%s.txt' % tissue, sep = '\t', index = False)

topx = 0.25
k=Network_Analysis(filename='Results_Paper/GCN_%s.txt' % tissue, network_type='transcriptomics',respath='Results_Paper/GCN_%s/' % tissue,
                   topx=topx, cluster_size = 30)
k.save_network()

In [ ]:
tissue = 'EF'
k = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))
color_mapping = dict(zip(sorted(k.clustering.unique()),sns.color_palette("tab10").as_hex()[::-1]))
x = []
for cl in sorted(k.clustering.unique()):
    k_temp = k.network.subgraph(k.clustering[k.clustering == cl].index)
    print(cl)
    print(k_temp.transitivity_avglocal_undirected())
    x.append(['%s-%d'% (tissue,cl), k_temp.transitivity_avglocal_undirected(), color_mapping[cl]])
x = pd.DataFrame(x)

plt.figure(figsize = (5,5))
ax = sns.barplot(data = x, x = 0, y = 1, palette = x[2])
ax.set_ylabel('Average Clustering Coefficients', fontsize = 15, fontweight = 'bold')
ax.set_xlabel('', fontsize = 15, fontweight = 'bold')
ax.set_xticklabels(ax.get_xticklabels(), fontsize = 15, fontweight = 'bold', rotation = 90, ha = 'center', va = 'top')
sns.despine(ax = ax)

x.to_csv('Results_Paper/GCN_%s/ClusteringCoefficient.txt' % tissue, sep = '\t')


plt.savefig('Results_Paper/GCN_%s/ClusteringCoefficient.pdf' % tissue, transparent = True)

## Downstream Network (Figure 3C, D)

In [ ]:
tissue = 'LV'
temp_metadata = (metadata[(metadata['Tissue'] == tissue)])
temp_tpm = tpm_all.loc[(~tpm_all.index.duplicated()), temp_metadata.index]

norm_tpm = pd.DataFrame(zscore(temp_tpm.T).T, index = temp_tpm.index, columns = temp_tpm.columns)#.rename(columns = conds["conds"].to_dict())

order = ['Ischemia','Both']


In [ ]:
deseq = pd.ExcelFile('Results_Paper/DifferentialExpression_ALL.xlsx')

res = pd.DataFrame()
for i in deseq.sheet_names:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['L2FC'].rename(i)
    res = pd.concat([res, temp], axis = 1)
res = res[res.columns[(res.columns.str.contains(tissue)) & (res.columns.str.contains('Ctrl'))  & (~res.columns.str.contains('Diabete'))]].dropna(how = 'all')
res.columns = [i.split('_')[0] if 'Ctrl' in i else "Both vs\nIschemia"  for i in res.columns]

In [ ]:
k = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

In [ ]:
KEGG = {i.split('\t')[0]:[j for j in i.rstrip('\n').split('\t')[2:] if j != ''] for i in open('../../../MainLibrary/Current/KEGG_2021_Human.gmt', 'r').readlines()}
GO = {i.split('\t')[0].split(' (GO')[0].replace('"', ''):[j for j in i.rstrip('\n').split('\t')[2:] if j != ''] for i in open('../../../MainLibrary/Current/GO_Biological_Process_2021.gmt', 'r').readlines()
     if (('positive' not in i) & ('negative' not in i))}

exclude_KEGG = ['Hepatitis B', 'Human T-cell leukemia virus 1 infection', 'Regulation of actin cytoskeleton', 'Measles',
           'MicroRNAs in cancer', 'Pathways in cancer', 'Influenza A', 'Oocyte meiosis', 'Progesterone-mediated oocyte maturation',
           'Ribosome biogenesis in eukaryotes', 'Amoebiasis', 'Small cell lung cancer', 'Epstein-Barr virus infection', 'Human papillomavirus infection',
           'Pertussis', 'Staphylococcus aureus infection', 'Proteoglycans in cancer', 'Leishmaniasis', 'Legionellosis', 'Viral carcinogenesis', 
           'Chagas disease (American trypanosomiasis)', 'Toxoplasmosis', 'Malaria', 'Metabolism of xenobiotics by cytochrome P450', 'Chronic myeloid leukemia',
           'Prion diseases', 'Inflammatory bowel disease (IBD)', 'Kaposi sarcoma-associated herpesvirus infection', 'Parkinson disease', 'Hepatitis C', 'Systemic lupus erythematosus',
           'Tuberculosis', 'Hepatitis B', 'Measles', 'Hematopoietic cell lineage', 
           'Epstein-Barr virus infection', 'Drug metabolism', 'Salivary secretion', 'Glioma', 'Human cytomegalovirus infection', 'Melanoma', 'Chemical carcinogenesis',
            'African trypanosomiasis', 'Collecting duct acid secretion', 'African trypanosomiasis', 
            'Rheumatoid arthritis', 'Metabolic pathways', 'Spinocerebellar ataxia', 'Drug metabolism - other enzymes', 'Asthma', 'Thyroid hormone synthesis',
                'Hepatocellular carcinoma', 'Axon guidance', 'Thermogenesis', 'Ribosome'
          ]

In [ ]:
color_mapping = dict(zip(sorted(k.clustering.unique()),sns.color_palette("tab10").as_hex()[::-1]))

fig, axs = plt.subplots(figsize = (8, len(k.clustering.unique())*2.1), nrows = len(k.clustering.unique()), ncols = 4, sharex = False,
                        gridspec_kw={'width_ratios':[.15,0.2, 0.2, 0.15]})

for cl in sorted(k.clustering.unique().astype(int)):
    sel_ax = axs[cl][0]
    sel_data = ((res.reindex(k.clustering[k.clustering == cl].index))).dropna(how = 'all')
    num = sel_data.shape[0]
    sel_data = sel_data.unstack().reset_index()
    sel_data = sel_data[sel_data[0].notna()]
    
    g = sns.barplot(data = sel_data, x = 'level_0', y = 0, order = order, ax = sel_ax, capsize = 0.2, errwidth= 1)
    if cl == k.clustering.max():
        g.set_xlabel('Log2FC', fontweight = 'bold', fontsize = 10)
        g.set_xticklabels(g.get_xticklabels(), fontweight = 'bold', fontsize = 10)
    else:
        g.set_xlabel('')
        g.set_xticks([])
    temp_ylim = tuple(g.get_ylim())
    temp_xlim = tuple(g.get_xlim())
    print(temp_ylim)
    g.hlines([0], xmin = g.get_xlim()[0], xmax = g.get_xlim()[1], color = 'k')
    g.set_xlim(temp_xlim)
    g.set_ylabel('%s-%d\n%d Genes' % (tissue, cl, k.clustering[k.clustering == cl].shape[0]), fontweight = 'bold', fontsize = 10, color = color_mapping[cl])
    
sel_term = []
for cl in sorted(k.clustering.unique().astype(int)):
    sel_ax = axs[cl][1]
    enr = gp.enrichr(gene_list=k.clustering[k.clustering == cl].index.tolist(),
             gene_sets=KEGG,
             background = temp_tpm.shape[0],
             outdir='test/enrichr_kegg2',
             cutoff=0.5,
             verbose=False)
    temp = enr.res2d[~(
        enr.res2d['Term'].isin(exclude_KEGG) |
        (enr.res2d['Term'].str.contains('isease')) |
         (enr.res2d['Term'].str.contains('arcinoma')) |
          (enr.res2d['Term'].str.contains('ancer')) |
        (enr.res2d['Term'].str.contains('thyroid')) |
        (enr.res2d['Term'].str.contains('Amyo')) |
        (enr.res2d['Term'].str.contains('yndrome')) |
        (enr.res2d['Term'].str.contains('strogen')) |
        (enr.res2d['Term'].str.contains('elanoge')) |
        (enr.res2d['Term'].str.contains('neuro')) |
        (enr.res2d['Term'].str.contains('iabet')) |
        (enr.res2d['Term'].str.contains('ysosome')) |
        (enr.res2d['Term'].str.contains('nfection')) |
        (enr.res2d['Term'].str.contains('homolog')) |
        (enr.res2d['Term'].str.contains('Proteasome')) |
        (enr.res2d['Term'].str.contains('ntestin')) |
        (enr.res2d['Term'].str.contains('cardi')) |
        (enr.res2d['Term'].str.contains('Cardi')) |
        (enr.res2d['Term'].str.contains('ubule')) |
        (enr.res2d['Term'].str.contains('Shigel')) |
        (enr.res2d['Term'].str.contains('nsulin')) |

        (enr.res2d['Term'].str.contains('Viral')) |
        (enr.res2d['Term'].str.contains('nemia')) |
        (enr.res2d['Term'].str.contains('mmunodef'))
    )]
    temp = temp.sort_values('P-value', ascending = True).iloc[0:5]
    temp2 = temp[['Term','Adjusted P-value']]
    sel_term.append(temp2['Term'].tolist())

    temp2['Term'] = temp2['Term'].str.replace('endoplasmic reticulum', 'ER')
    temp2['-Log10(P-Adj)'] = -np.log10(temp2['Adjusted P-value'])
    sns.barplot(data = temp2, x = '-Log10(P-Adj)', y = 'Term', ax = sel_ax, color = color_mapping[cl])
    sel_ax.yaxis.tick_right()
    sel_ax.set_yticklabels([i for i in temp2['Term']], fontsize = 10)
    # sel_ax.set_xticks([0])
    if cl == len(x) - 1:
        sel_ax.set_xlabel('-Log10(P-Adj)', fontweight = 'bold', fontsize = 10)
        sel_ax.set_xticks([])
        sel_ax.set_xticklabels([], fontweight = 'bold', fontsize = 10)
    else:
        sel_ax.set_xlabel('')
        sel_ax.set_xticks([])
    sns.despine(ax = sel_ax, left = False, bottom = False, right = True, top = True)
    sel_ax.set_ylabel('')
    sel_ax.tick_params(right=False, bottom=False)
    
for cl in sorted(k.clustering.unique().astype(int)):
    axs[cl][2].axis('off')
    
for cl in sorted(k.clustering.unique().astype(int)):
    sel_ax = axs[cl][3]
    topx = 10
    temp = pd.DataFrame()
  cluster = pd.concat([k.clustering, k.degree], axis = 1).reindex(res.index).dropna().reset_index()
    cluster = cluster[cluster[0] == cl].sort_values('degree', ascending = False).iloc[0:topx]
    sns.scatterplot(data = cluster, x = 'degree', y = 'index', ax = sel_ax, color = color_mapping[cl], legend = False)
    sel_ax.yaxis.tick_right()
    sel_ax.set_yticklabels([i for i in cluster['index']], fontsize = 10)
    # sel_ax.set_xticks([0])
    if cl == k.clustering.max():
        sel_ax.set_xlabel('Top Nodes\n(Degree)', fontweight = 'bold', fontsize = 10)
        sel_ax.set_xticks([])
        sel_ax.set_xticklabels([], fontweight = 'bold', fontsize = 10)
    else:
        sel_ax.set_xlabel('')
        sel_ax.set_xticks([])
    sns.despine(ax = sel_ax, left = False, bottom = False, right = True, top = True)
    sel_ax.set_ylabel('')
    sel_ax.tick_params(right=False, bottom=False)
plt.savefig('Results_Paper/GCN_%s/Cluster_Trends.pdf' % tissue, transparent = True)

## Integration

### Load Network

In [ ]:
tissue = 'LV'
k_LV = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))
cluster_LV = pd.concat([k_LV.clustering.rename('Cluster_%s' % tissue), k_LV.degree.rename('Degree_%s' % tissue)], axis = 1)

tissue = 'EF'
k_EF = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))
cluster_EF = pd.concat([k_EF.clustering.rename('Cluster_%s' % tissue), k_EF.degree.rename('Degree_%s' % tissue)], axis = 1)

color_mapping = dict(zip(sorted(k_EF.clustering.unique()),sns.color_palette("tab10").as_hex()[::-1]))
color_mapping

deseq = pd.ExcelFile('Results_Paper/DifferentialExpression_ALL.xlsx')

res = pd.DataFrame()
for i in deseq.sheet_names:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['L2FC'].rename(i)
    res = pd.concat([res, temp], axis = 1)
res = res[res.columns[(~res.columns.str.contains('Diabete'))]].dropna(how = 'all')
# res = res[res.columns[res.columns.str.contains('None') & (~res.columns.str.contains('Diabete'))]].dropna(how = 'all')

all_clust = pd.concat([cluster_LV, cluster_EF], axis = 1).dropna()
deg_clust = pd.concat([res, all_clust.reindex(res.index)], axis = 1)


In [ ]:
k_LV.network.density()

### Intersection (Figure 4B)

In [ ]:
num_ofpw = all_clust.shape[0]
y = []
for cl in sorted(all_clust['Cluster_LV'].unique()):
    temp = all_clust['Cluster_LV'][all_clust['Cluster_LV'] == cl]
    for node in sorted(all_clust['Cluster_EF'].unique()):
        hits_an = set(temp.index.str.upper()).intersection(all_clust['Cluster_EF'][all_clust['Cluster_EF'] == node].index)
        hits = len(hits_an)
        num_member_inpw = all_clust['Cluster_EF'][all_clust['Cluster_EF'] == node].shape[0]
        pval = 1-hypergeom.cdf(hits,num_ofpw-num_member_inpw, temp.shape[0],num_member_inpw)
        y.append(['EF-%d' % node, 'LV-%d' % cl, pval])
results_cl = pd.DataFrame(y, columns = ['EF', 'LV', 'pval'])
results_cl['padj']=multipletests(results_cl['pval'],method='fdr_bh')[1]
results_cl = results_cl[results_cl['padj'] < 0.05].sort_values(['EF'], ascending = False)
results_cl['-Log10(P-Adj)'] = -np.log10(results_cl['padj'])
results_cl.replace(np.inf, 10).to_csv('Results_Paper/Intersection_Network.txt', sep = '\t')

In [ ]:
results_cl

In [ ]:
temp = results_cl.pivot_table(index = 'EF', columns = 'LV', values = '-Log10(P-Adj)').fillna(0)
temp[temp != 0] = 1
temp['logFC'] = 1

temp.to_csv('Results_Paper/Intersection_Network_chord.txt', sep = '\t')

In [ ]:
results_cl['annot'] = ['*' if i < 0.05 else '' for i in results_cl['padj']]

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
library(circlize)

data = read.csv('Results_Paper/Intersection_Network.txt', sep = '\t')
data
adjacencyData <- with(data, table(EF, LV))

grid.col = c(
  'EF-0' = "#17becf", 'LV-0' = "#17becf",
  'EF-1' = "#bcbd22", 'LV-1' = "#bcbd22",
  'EF-2' = "#7f7f7f", 'LV-2' = "#7f7f7f",
  'EF-3' = "#e377c2", 'LV-3' = "#e377c2",
  'EF-4' = "#8c564b", 'LV-4' = "#8c564b",
    'EF-5' = "#9467bd",
    'EF-6' = "#d62728",
    'EF-7' = "#2ca02c"
)

pdf("Results_Paper/FiguresPaper/4B.pdf") 
circos.par(start.degree = 90)
chordDiagram(adjacencyData, transparency = 0.25,
             annotationTrack = c("name", "grid"),
             order = c('EF-7', 'EF-6', 'EF-5', 'EF-4', 'EF-3', 'EF-1', 'EF-0','LV-4', 'LV-3', 'LV-2', 'LV-1', 'LV-0'),
             grid.col = grid.col#, col = data$link
)
dev.off()


### TF (Figure 4C)

In [ ]:
tpm_all = supp_data.parse('TPM', index_col = 0)

In [ ]:
TF = {i.split('\t')[0]:list(set([j for j in i.rstrip('\n').split('\t')[2:] if j != '']).intersection(tpm_all.index)) for i in open('Library/CHEATranscriptionFactorTargets.gmt', 'r').readlines()}


In [ ]:
y = []

num_ofpw = tpm_all.shape[0]
for cl_LV in sorted(k_LV.clustering.unique()):
    temp = k_LV.clustering[k_LV.clustering == cl_LV]
    for gse in TF.keys():
        hits_an = set(temp.index).intersection(TF[gse])#.intersection(res[res.columns[res.columns.str.contains('LV')]].dropna(how = 'all').index)
        degs = hits_an.intersection(res[res.columns[res.columns.str.contains('LV')]].dropna(how = 'all').index)
        hits = len(hits_an)
        num_member_inpw = len(TF[gse])
        pval = 1-hypergeom.cdf(hits,num_ofpw-num_member_inpw, temp.shape[0],num_member_inpw)
        y.append([gse, 'LV-%d' % cl_LV, pval, hits, len(TF[gse]), len(degs), 'LV'])

results_cl = pd.DataFrame(y, columns = ['TF', 'cluster', 'pval', 'hits', 'total', '#degs', 'type']).fillna(1)
results_cl['padj']=multipletests(results_cl['pval'],method='fdr_bh')[1]

for cl_EF in sorted(k_EF.clustering.unique()):
    temp = k_EF.clustering[k_EF.clustering == cl_EF]
    for gse in TF.keys():
        hits_an = set(temp.index).intersection(TF[gse])#.intersection(res[res.columns[res.columns.str.contains('EF')]].dropna(how = 'all').index)
        degs = hits_an.intersection(res[res.columns[res.columns.str.contains('EF')]].dropna(how = 'all').index)
        hits = len(hits_an)
        num_member_inpw = len(TF[gse])
        pval = 1-hypergeom.cdf(hits,num_ofpw-num_member_inpw, temp.shape[0],num_member_inpw)
        # if len(degs) < 20:
        #     continue
        y.append([gse, 'EF-%d' % cl_EF, pval, hits, len(TF[gse]), len(degs), 'EF'])

results_cl = pd.DataFrame(y, columns = ['TF', 'cluster', 'pval', 'hits', 'total', '#degs', 'type']).dropna()
results_cl['padj']=multipletests(results_cl['pval'],method='fdr_bh')[1]


results_cl = results_cl[(results_cl['padj'] < 0.05)].sort_values(['TF'], ascending = False)
temp=results_cl[results_cl['cluster'].isin(['LV-2', 'LV-3', 'EF-3', 'EF-5'])].sort_values(['TF', 'cluster', '#degs'])
cTF = ['MITF', 'SRF','GATA1', 'GATA2', 'GATA3', 'GATA4', 'MEF2A', 'TEAD4', 'FOXO3', 'YY1', 'TBX5', 'JARID2', ]

results_cl[results_cl['TF'].isin(cTF)].to_csv('Results_Paper/TF_Connection_degs_Heart.txt', sep = '\t')
results_cl.to_csv('Results_Paper/TF_Connection_degs_ALL.txt', sep = '\t')

In [ ]:
temp = results_cl[results_cl['#degs'] >= 0].pivot_table(columns = 'cluster', index = 'TF', values = 'padj')[['LV-2', 'EF-3']].dropna(thresh = 2)
temp = (-np.log10(temp)).fillna(0).replace(np.inf, 10)
degs_num = results_cl[results_cl['#degs'] >= 0].pivot_table(columns = 'cluster', index = 'TF', values = '#degs')[['LV-2', 'EF-3']].dropna(thresh = 2)
degs_num[degs_num.notna()] = '*'

g = sns.clustermap(temp.fillna(0), figsize = (4, 15), cmap = 'Reds', yticklabels = 1, col_cluster=False, annot = degs_num.fillna(''), fmt = 's')
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/S4.pdf')

In [ ]:
temp = results_cl[results_cl['TF'].isin(cTF) & (results_cl['#degs'] >= 0)].pivot_table(columns = 'cluster', index = 'TF', values = 'padj')[['LV-0','LV-1','LV-2', 'LV-3', 'EF-0', 'EF-1', 'EF-2', 'EF-3', 'EF-5', 'EF-6']].dropna(thresh = 2)
temp = (-np.log10(temp)).fillna(0).replace(np.inf, 10)
degs_num = results_cl[results_cl['TF'].isin(cTF) & (results_cl['#degs'] >= 0)].pivot_table(columns = 'cluster', index = 'TF', values = '#degs')[['LV-0','LV-1','LV-2', 'LV-3', 'EF-0', 'EF-1', 'EF-2', 'EF-3', 'EF-5', 'EF-6']].dropna(thresh = 2)
# degs_num[degs_num.notna()] = '*'
g = sns.clustermap(temp.fillna(0), figsize = (4, 3), cmap = 'Reds', yticklabels = 1, col_cluster=False, annot = degs_num)#.fillna(''), fmt = 's')
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
g.savefig('Results_Paper/FiguresPaper/4C.pdf')

## Clusters to Metabolite (Figure 4A)

Visualized in cytoscape

In [ ]:
HMDB_gmt = {i.split('\t')[0]:list(set([j for j in i.rstrip('\n').split('\t')[2:] if j != '']).intersection(tpm_all.index)) for i in open('Library/HMDB_Metabolites.gmt', 'r').readlines()
           }


In [ ]:
tissue = 'LV'
k_LV = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

tissue = 'EF'
k_EF = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

In [ ]:
mets = {}
for cl_LV in set(k_LV.clustering.unique()):
    enr = gp.enrichr(gene_list=k_LV.clustering[k_LV.clustering == cl_LV].index.tolist(),
             gene_sets=HMDB_gmt,
             background = tpm_all.index.tolist(),
             outdir='test/enrichr_kegg2',
             cutoff=0.5,
             verbose=False)
    mets['LV-%d' % cl_LV] = enr.res2d[enr.res2d['Adjusted P-value'] < 0.05].set_index('Term')[['Overlap', 'P-value', 'Adjusted P-value']]#['Term'].tolist()

In [ ]:
for cl_EF in set(k_EF.clustering.unique()):
    enr = gp.enrichr(gene_list=k_EF.clustering[k_EF.clustering == cl_EF].index.tolist(),
             gene_sets=HMDB_gmt,
             background = tpm_all.index.tolist(),
             outdir='test/enrichr_kegg2',
             cutoff=0.5,
             verbose=False)
    mets['EF-%d' % cl_EF] = enr.res2d[enr.res2d['Adjusted P-value'] < 0.05].set_index('Term')[['Overlap', 'P-value', 'Adjusted P-value']]#['Term'].tolist()

In [ ]:
pd.concat(mets, axis = 1, keys = mets.keys()).to_excel('Results_Paper/RM.xlsx')

In [ ]:
x = []
for i in mets.keys():
    temp = list(zip([i]*len(mets[i]),[k.split(' (HM')[0] for k in mets[i]]))
    x.append(temp)
    
x = pd.DataFrame(sum(x, []), columns = ['source','target'])
x.to_csv('Results_Paper/RM_edges.txt',sep= '\t')

In [ ]:
for i in sorted(set(mets['LV-1']).intersection(mets['EF-3'])):
    # if '-' in i:
    #     continue
    print(i.split(' (')[0])

In [ ]:
cl_LV = 1
t1 = k_LV.clustering[k_LV.clustering == cl_LV].reindex(heart_specific.index).dropna()
cl_LV = 3
t2 = k_EF.clustering[k_EF.clustering == cl_LV].reindex(heart_specific.index).dropna()

In [ ]:
for i in set(t1.index).intersection(t2.index):
    if i in fda.index:
        print(i)

In [ ]:
', '.join(k_LV.degree.reindex(t1.index).reindex(fda.index).dropna().sort_values().index)

In [ ]:
x = set()
for i in sorted(set(mets['LV-1']).intersection(mets['EF-3'])):
    print(i)

In [ ]:
set(HMDB_gmt['Ubiquinone Q1 (HMDB02012)'] + HMDB_gmt['Coenzyme Q (HMDB06709)'] + HMDB_gmt['QH2 (HMDB01304)'])

## Finding Targets

In [ ]:
tpm_all = supp_data.parse('TPM', index_col = 0)

In [ ]:
TF = {i.split('\t')[0]:list(set([j for j in i.rstrip('\n').split('\t')[2:] if j != '']).intersection(tpm_all.index)) for i in open('Library/CHEATranscriptionFactorTargets.gmt', 'r').readlines()}
hpa = pd.read_csv('Library/proteinatlas.tsv', sep = '\t', index_col = 0)
fda = hpa[hpa['Disease involvement'].astype(str).str.contains('FDA')]['Disease involvement']
heart_specific = hpa[hpa['RNA tissue specificity'].isin(['Tissue enhanced', 'Group enriched','Tissue enriched']) & (hpa['RNA tissue specific nTPM'].str.contains('heart') | hpa['RNA tissue specific nTPM'].str.contains('adipose'))]
HMDB_gmt = {i.split('\t')[0]:list(set([j for j in i.rstrip('\n').split('\t')[2:] if j != '']).intersection(tpm_all.index)) for i in open('Library/HMDB_Metabolites.gmt', 'r').readlines()
           }

In [ ]:
tissue = 'LV'
k_LV = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

tissue = 'EF'
k_EF = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

In [ ]:
deseq = pd.ExcelFile('Results_Paper/DifferentialExpression_ALL.xlsx')
res = pd.DataFrame()
lst = ['Ischemia_EF_Ctrl_EF', 'Both_EF_Ctrl_EF','Ischemia_LV_Ctrl_LV', 'Both_LV_Ctrl_LV']
for i in lst:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['DIRECTION'].rename(i)
    res = pd.concat([res, temp], axis = 1)
    
deseq = pd.ExcelFile('Results_Paper/Validation/DifferentialExpression_ALL.xlsx')
lst = deseq.sheet_names
for i in lst:
    temp = deseq.parse(i, index_col = 0)
    temp = temp[temp['P-VALUE'] < 0.05]['DIRECTION'].rename(i)
    res = pd.concat([res, temp], axis = 1)


In [ ]:
val = res[['Ischemia_LV_Ctrl_LV', 'Both_LV_Ctrl_LV', 'ischemic_cardiomyopathy_non_fai']].dropna()
val = val[(val['Ischemia_LV_Ctrl_LV'] == val['ischemic_cardiomyopathy_non_fai']) | (val['ischemic_cardiomyopathy_non_fai'] == val['Both_LV_Ctrl_LV'])]

In [ ]:
sel_tf = ['GATA4', 'SRF', 'TBX5', 'YY1','MEF2A']
sel_rm = ['Iron (HMDB00692)', 'Sulfide (HMDB00598)', 'NAD (HMDB00902)', 'NADH (HMDB01487)','FAD (HMDB01197)', 
                                         'FAD (HMDB01248)','Ubiquinone Q1 (HMDB02012)', 'Coenzyme Q (HMDB06709)', 'QH2 (HMDB01304)']

In [ ]:
gl_fda = list(zip(fda.index, ['FDA Approved'] * fda.shape[0]))
gl_heart = list(zip(heart_specific.index, ['Heart Specific/Enriched'] * fda.shape[0]))
gl_tf = sum([list(zip(TF[i], [i]*len(TF[i]))) for i in sel_tf], [])
gl_repmets = sum([list(zip(HMDB_gmt[i], [i]*len(HMDB_gmt[i]))) for i in sel_rm], [])

In [ ]:
temp1 = pd.DataFrame(gl_fda + gl_heart + gl_tf + gl_repmets, columns = ['index',0])
temp2 = k_LV.clustering[k_LV.clustering == 2].reset_index()
temp2[0] = 'LV-' + temp2[0].astype(str)
temp3 = k_EF.clustering[k_EF.clustering == 3].reset_index()
temp3[0] = 'EF-' + temp3[0].astype(str)
final = pd.concat([temp1, temp2, temp3])
final['values'] = 1
final = final.pivot_table(index = 0, columns = 'index', values = 'values').T#.dropna(thresh = 3)

final = final[
    ((final['LV-2'] == 1) & (final['EF-3'] == 1)) & #Must be in LV-2 and EF-3
    # (final['Heart Specific/Enriched'] == 1) & # Must be Heart Specific
    # (final['FDA Approved'] == 1) & # Must be FDA Approved Targets
    (
        (final[sel_tf].sum(1) > 0) #Must be in regulated by one of the selected TF ('GATA4', 'SRF', 'TBX5', 'YY1','MEF2A')
        & (final[sel_rm].sum(1) > 0) #Must be associated with the selceted reporter metabolites (sel_rm variable)
    )
]
final = final[['LV-2', 'EF-3', 'Heart Specific/Enriched', 'FDA Approved'] + sel_rm + sel_tf].sort_values(['FDA Approved', 'Heart Specific/Enriched'] + sel_rm + sel_tf)#.to_excel('Results_Paper/Final_Genes.xlsx')

In [ ]:
for i in final.index:
    print(i)

## SDHA in CNs

In [ ]:
tissue = 'LV'
k_LV = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

tissue = 'EF'
k_EF = pickle.load(open('Results_Paper/GCN_%s/network_object.pkl' % tissue, 'rb'))

In [ ]:
k_LV.network_ori[(k_LV.network_ori['level_0'].isin(['SDHA', 'OGDH'])) | (k_LV.network_ori['level_1'].isin(['SDHA', 'OGDH']))].to_csv('Results_Paper/SDHA_OGDH_LV.txt', sep = '\t', index = False)
k_EF.network_ori[(k_EF.network_ori['level_0'].isin(['SDHA', 'OGDH'])) | (k_EF.network_ori['level_1'].isin(['SDHA', 'OGDH']))].to_csv('Results_Paper/SDHA_OGDH_EF.txt', sep = '\t', index = False)

## Check Correlation

In [ ]:
data = supp_data.parse('Clinical', index_col = 0)

In [ ]:
lst = data.columns[1:]

In [ ]:
tissue = 'LV'

gene = 'SDHA'
temp = pd.concat([tpm_all.loc[gene], metadata[['Tissue', 'Code', 'Condition']]], axis = 1)
temp_x = temp[temp['Tissue'] == tissue].merge(data, left_on = 'Code', right_index = True)
# temp_x = temp_x[temp_x['Condition'] != 'Ctrl'].sort_values(gene)

x = []

for i in temp_x[gene]:
    if i < temp_x[gene].quantile(1/3):
        x.append('low')
    elif i > temp_x[gene].quantile(2/3):
        x.append('high')
    else:
        x.append('mid')

temp_x['level'] = x

table1 = pd.DataFrame()
for i in lst:
    temp = temp_x[[i, 'level']].dropna()
    temp = temp[temp['level'] != 'mid']
    # if temp.shape[0] < 6:
    #     continue
    mean = temp.groupby('level')[i].mean()
    sem = temp.groupby('level')[i].sem()
    fin = (mean.apply(lambda x: "%.2f" % x) + u" \u00B1 " +sem.apply(lambda x: "%.2f" % x))[['low', 'high']]
    ttest = [ttest_ind(temp[temp['level'] == 'high'][i],temp[temp['level'] == 'low'][i])[1]]
        
    if ttest[0] <= 0.0005:
        sig = '***'
    elif ttest[0] <= 0.005:
        sig = '**'
    elif ttest[0] <= 0.05:
        sig = '*'
    else:
        sig = 'ns'
    fin['P-Value\n(high vs low)'] = ttest[0]#sig

    table1 = pd.concat([table1, fin], axis = 1).fillna('ns')
table1 = table1.T.sort_values('P-Value\n(high vs low)')
table1['P-Adj\n(high vs low)'] = multipletests(table1['P-Value\n(high vs low)'],method='fdr_bh')[1]

In [ ]:
hi = table1[table1["P-Adj\n(high vs low)"] < 0.05].index.tolist()
comp = list(zip([gene]*len(hi), hi))
fig, axs = plt.subplots(figsize = (12, 10),nrows = 3, ncols = 4, sharex = False)
axs = axs.flatten()
for i, j in enumerate(comp):
    print(i)
    g = axs[i]
    x, y = j
    temp = temp_x[[x, y, 'level']].dropna()
    temp = temp[temp['level'] != 'mid']

    sns.boxplot(data = temp, x = 'level', y = y, ax = g, order = ['low', 'high'], color = 'gray')
    sns.stripplot(data = temp, x = 'level', y = y, ax = g, order = ['low', 'high'], dodge = True, color = 'black')
    add_stat_annotation(data = temp, x = 'level', y = y, order=['low', 'high'], ax = g,
                    box_pairs=[('low', 'high')],
                    perform_stat_test=True,# pvalues=pval_sel,
                    test='t-test_ind', text_format='star', loc='inside', verbose=0, fontsize = 15)
    sns.despine(ax = g)
    g.set_title(g.get_ylabel(), fontsize = 15, fontweight = 'bold', style = 'italic')
    g.set_ylabel('', fontsize = 15, fontweight = 'bold', style = 'italic')
    g.set_xlabel('%s Level (%s)' % (gene,tissue), fontsize = 15, fontweight = 'bold', style = 'italic')
plt.subplots_adjust(wspace=0.5,hspace=0.3)
plt.savefig('Results_Paper/FiguresPaper/5B-C_%s.pdf' % gene)